# Beginner Tutorial: 12AX with btorch GLIF-RSNN

This notebook rebuilds a compact 12AX training pipeline in notebook cells while keeping btorch-critical steps explicit.

What you will build:
- dataclass-first OmegaConf config with notebook-safe overrides
- 12AX data pipeline with Poisson spike encoding
- full `SingleLayerGLIFRSNN` class definition inlined (model, recurrent wiring, checkpointing)
- learnable input scaling trajectory analysis
- checkpoint save/load with `memories_rv` and runtime `memory_values`

What you should observe:
- shape contracts stay consistent from data to logits
- scale parameter changes during optimization
- checkpoint contains memory reset values and runtime memory snapshots


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from btorch.models import functional

sys.path.append("../..")

from glif_net.src.checkpoint import load_checkpoint, save_checkpoint
from tutorial_12ax_utils import (
    SYMBOLS,
    aggregate_symbol_logits,
    compute_targets,
    generate_batch,
    generate_symbol_sequence,
    plot_scale_history,
    plot_spike_raster,
    plot_symbol_target_timeline,
    plot_training_curves,
)

plt.rcParams["figure.dpi"] = 110


## 1) Environment setup
Deterministic seed + device selection.

In [ ]:
def set_seed(seed: int) -> np.random.Generator:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    return np.random.default_rng(seed)

# Default to CPU for notebook stability; switch to GPU if you have >=4 GB VRAM
device = torch.device("cpu")
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2) Dataclass + OmegaConf pattern
The config is notebook-local and keeps key fields aligned with `task_12ax.py`.

In [ ]:
@dataclass
class Config:
    seed: int = 7
    profile: str = "quick"  # quick | paper
    dt: float = 1.0

    episode_symbols: int = 90
    symbol_ms: int = 500
    input_neurons_per_symbol: int = 5
    poisson_rate_hz: float = 200.0

    n_neuron: int = 200
    n_adapt: int = 100
    n_e_ratio: float = 0.8
    tau_mem: float = 20.0
    tau_syn: float = 5.0
    tau_ref: float = 5.0
    v_threshold: float = -30.0
    v_reset: float = -60.0
    asc_amp: float = -0.00425
    tau_adapt_min: float = 1.0
    tau_adapt_max: float = 13500.0

    iterations: int = 10_000
    batch_size: int = 20
    lr: float = 1e-3
    max_grad_norm: float = 1.0
    grad_checkpoint: bool = True
    chunk_size: int = 1000

    rate_reg_coeff: float = 15.0
    target_rate_hz: float = 10.0

    eval_every: int = 200
    eval_batches: int = 10


def apply_profile_overrides(cfg: Config) -> Config:
    if cfg.profile == "quick":
        cfg.symbol_ms = min(cfg.symbol_ms, 60)
        cfg.iterations = min(cfg.iterations, 60)
        cfg.batch_size = min(cfg.batch_size, 2)
        cfg.n_neuron = min(cfg.n_neuron, 100)
        cfg.eval_every = min(cfg.eval_every, 20)
        cfg.eval_batches = min(cfg.eval_batches, 2)
    return cfg

base = Config()
cfg_obj = apply_profile_overrides(base)

print("Resolved config:")
pd.Series(asdict(cfg_obj))


## 3) 12AX data pipeline (lean walkthrough)
Helpers live in `tutorial_12ax_utils.py`; the notebook keeps only usage-level logic and shape checks.

In [ ]:
rng = set_seed(cfg_obj.seed)

sample_sequence = generate_symbol_sequence(cfg_obj.episode_symbols, rng)
sample_targets = compute_targets(sample_sequence)

x_batch, y_batch = generate_batch(cfg_obj, rng, device)
expected_input_dim = len(SYMBOLS) * cfg_obj.input_neurons_per_symbol
assert x_batch.shape == (cfg_obj.episode_symbols * cfg_obj.symbol_ms, cfg_obj.batch_size, expected_input_dim)
assert y_batch.shape == (cfg_obj.episode_symbols, cfg_obj.batch_size)

print("x shape (T,B,input_dim):", tuple(x_batch.shape))
print("y shape (S,B):", tuple(y_batch.shape))
print("first 24 symbols:", "".join(sample_sequence[:24]))
print("R count in sample episode:", int((sample_targets == 1).sum().item()))


## 4) Plot sampled symbols and targets
R labels appear only for context-consistent `AX` (under `1`) and `BY` (under `2`).

In [ ]:
plot_symbol_target_timeline(sample_sequence, sample_targets, max_symbols=80)


## 5) `SingleLayerGLIFRSNN` — full class definition

Key btorch APIs:
- `btorch.models.environ.context(dt=...)` wraps the recurrent forward pass
- `btorch.models.rnn.RecurrentNN(..., grad_checkpoint=..., chunk_size=...)` manages the recurrent simulation with optional gradient checkpointing
- `btorch.models.linear.LearnableScale` wraps input and output for learnable gain
- `btorch.models.linear.SparseConn(..., enforce_dale=True)` enforces Dale's law on recurrent weights
- `btorch.models.init.uniform_v_` initializes voltage states and sets deterministic reset values

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import scipy.sparse
import torch
import torch.nn as nn

from btorch.models import environ, functional
from btorch.models.init import uniform_v_
from btorch.models.linear import LearnableScale, SparseConn
from btorch.models.neurons import GLIF3
from btorch.models.rnn import RecurrentNN
from btorch.models.synapse import ExponentialPSC


@dataclass(frozen=True)
class ConnectivitySpec:
    density: float = 1.0
    i_e_ratio: float = 100.0
    e_to_e_mean: float = 4.0e-3
    e_to_e_std: float = 1.9e-3
    e_i_mean: float = 5.0e-2
    i_i_mean: float = 25e-4


def _apply_density(
    rows: np.ndarray,
    cols: np.ndarray,
    vals: np.ndarray,
    density: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if density >= 1.0:
        return rows, cols, vals
    keep = int(rows.size * density)
    if keep <= 0:
        return rows[:0], cols[:0], vals[:0]
    idx = rng.choice(rows.size, keep, replace=False)
    return rows[idx], cols[idx], vals[idx]


def build_sparse_mat(
    n_e_neurons: int,
    n_i_neurons: int,
    split: bool = False,
    density: float = 1.0,
    i_e_ratio: float = 100.0,
    e_to_e_mean: float = 4.0e-3,
    e_to_e_std: float = 1.9e-3,
    e_i_mean: float = 5.0e-2,
    i_i_mean: float = 25e-4,
    seed: int | None = None,
) -> tuple[scipy.sparse.coo_array, np.ndarray, np.ndarray]:
    if not 0.0 <= density <= 1.0:
        raise ValueError("density must be between 0 and 1")
    if n_e_neurons <= 0 or n_i_neurons < 0:
        raise ValueError("invalid neuron counts")

    rng = np.random.default_rng(seed)
    n_neurons = n_e_neurons + n_i_neurons
    e_idx = np.arange(n_e_neurons)
    i_idx = np.arange(n_e_neurons, n_neurons)

    all_rows: list[np.ndarray] = []
    all_cols: list[np.ndarray] = []
    all_vals: list[np.ndarray] = []

    e_rows_list: list[np.ndarray] = []
    e_cols_list: list[np.ndarray] = []
    e_vals_list: list[np.ndarray] = []
    i_rows_list: list[np.ndarray] = []
    i_cols_list: list[np.ndarray] = []
    i_vals_list: list[np.ndarray] = []

    m = np.log(e_to_e_mean**2 / np.sqrt(e_to_e_std**2 + e_to_e_mean**2))
    s = np.sqrt(np.log(1.0 + (e_to_e_std**2 / e_to_e_mean**2)))
    e_to_e_weights = np.exp(rng.normal(loc=m, scale=s, size=(n_e_neurons, n_e_neurons)))

    e_rows, e_cols = np.meshgrid(e_idx, e_idx, indexing="ij")
    rows = e_rows.flatten()
    cols = e_cols.flatten()
    vals = e_to_e_weights.flatten()
    rows, cols, vals = _apply_density(rows, cols, vals, density, rng)
    all_rows.append(rows)
    all_cols.append(cols)
    all_vals.append(vals)
    e_rows_list.append(rows)
    e_cols_list.append(cols)
    e_vals_list.append(vals)

    mean_e_weights = e_to_e_weights.mean(axis=0)
    mean_i_weights = mean_e_weights * i_e_ratio
    for e_neuron in e_idx:
        vals = np.abs(
            rng.normal(
                loc=mean_i_weights[e_neuron],
                scale=max(mean_i_weights[e_neuron] * 0.25, 1e-12),
                size=n_i_neurons,
            )
        )
        rows = i_idx.copy()
        cols = np.full(n_i_neurons, e_neuron)
        rows, cols, vals = _apply_density(rows, cols, vals, density, rng)
        all_rows.append(rows)
        all_cols.append(cols)
        all_vals.append(-vals)
        i_rows_list.append(rows)
        i_cols_list.append(cols)
        i_vals_list.append(vals)

    e_rows, i_cols = np.meshgrid(e_idx, i_idx, indexing="ij")
    rows = e_rows.flatten()
    cols = i_cols.flatten()
    vals = np.full(rows.size, e_i_mean, dtype=np.float64)
    rows, cols, vals = _apply_density(rows, cols, vals, density, rng)
    all_rows.append(rows)
    all_cols.append(cols)
    all_vals.append(vals)
    e_rows_list.append(rows)
    e_cols_list.append(cols)
    e_vals_list.append(vals)

    i_rows, i_cols = np.meshgrid(i_idx, i_idx, indexing="ij")
    rows = i_rows.flatten()
    cols = i_cols.flatten()
    vals = np.full(rows.size, i_i_mean, dtype=np.float64)
    rows, cols, vals = _apply_density(rows, cols, vals, density, rng)
    all_rows.append(rows)
    all_cols.append(cols)
    all_vals.append(-vals)
    i_rows_list.append(rows)
    i_cols_list.append(cols)
    i_vals_list.append(vals)

    full_matrix = scipy.sparse.coo_array(
        (
            np.concatenate(all_vals),
            (np.concatenate(all_rows), np.concatenate(all_cols)),
        ),
        shape=(n_neurons, n_neurons),
    )

    if not split:
        return full_matrix, e_idx, i_idx

    e_matrix = scipy.sparse.coo_array(
        (
            np.concatenate(e_vals_list),
            (np.concatenate(e_rows_list), np.concatenate(e_cols_list)),
        ),
        shape=(n_neurons, n_neurons),
    )
    i_matrix = scipy.sparse.coo_array(
        (
            np.concatenate(i_vals_list),
            (np.concatenate(i_rows_list), np.concatenate(i_cols_list)),
        ),
        shape=(n_neurons, n_neurons),
    )
    return e_matrix, i_matrix, e_idx, i_idx


def sparseconn_dale_violations(
    sparse_conn: torch.nn.Module,
    n_e_neurons: int,
) -> dict[str, float]:
    if not hasattr(sparse_conn, "magnitude") or not hasattr(sparse_conn, "indices"):
        return {"count": 0.0, "fraction": 0.0}
    src_idx = sparse_conn.indices[1]
    w = sparse_conn.magnitude.detach()
    excitatory = src_idx < n_e_neurons
    bad_e = (w[excitatory] < 0).sum().item()
    bad_i = (w[~excitatory] > 0).sum().item()
    bad = float(bad_e + bad_i)
    return {
        "count": bad,
        "fraction": bad / float(max(w.numel(), 1)),
    }


class SingleLayerGLIFRSNN(nn.Module):
    """Single recurrent GLIF layer with readout and adaptation currents."""

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        n_neuron: int = 300,
        n_e_ratio: float = 0.8,
        n_adapt: int = 120,
        asc_amp: float = -0.2,
        tau_adapt: float = 700.0,
        tau_adapt_min: float | None = None,
        tau_adapt_max: float | None = None,
        tau: float = 20.0,
        tau_syn: float = 5.0,
        v_threshold: float = -45.0,
        v_reset: float = -60.0,
        tau_ref: float | None = 0.0,
        input_scale: float = 1.0,
        output_scale: float = 1.0,
        response_window: float = 0.8,
        readout_tau: float = 20.0,
        dt: float = 1.0,
        connectivity_density: float = 1.0,
        i_e_ratio: float = 100.0,
        e_to_e_mean: float = 4.0e-3,
        e_to_e_std: float = 1.9e-3,
        e_i_mean: float = 5.0e-2,
        i_i_mean: float = 25e-4,
        conn_seed: int | None = None,
        grad_checkpoint: bool = False,
        unroll: int | bool = 8,
        chunk_size: int | None = None,
        cpu_offload: bool = False,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_neuron = n_neuron
        self.n_adapt = n_adapt
        self.response_window = response_window
        self.readout_tau = readout_tau
        self.dt = float(dt)

        self.n_e = int(round(n_neuron * n_e_ratio))
        self.n_i = n_neuron - self.n_e
        if self.n_e <= 0 or self.n_i < 0:
            raise ValueError("invalid E/I split")

        self.connectivity_spec = ConnectivitySpec(
            density=connectivity_density,
            i_e_ratio=i_e_ratio,
            e_to_e_mean=e_to_e_mean,
            e_to_e_std=e_to_e_std,
            e_i_mean=e_i_mean,
            i_i_mean=i_i_mean,
        )
        weights, self.e_idx, self.i_idx = build_sparse_mat(
            n_e_neurons=self.n_e,
            n_i_neurons=self.n_i,
            density=connectivity_density,
            i_e_ratio=i_e_ratio,
            e_to_e_mean=e_to_e_mean,
            e_to_e_std=e_to_e_std,
            e_i_mean=e_i_mean,
            i_i_mean=i_i_mean,
            seed=conn_seed,
        )

        self.input_linear = nn.Linear(input_dim, n_neuron, bias=False)
        self.input_scale = LearnableScale(scale=input_scale, bias=None)

        self.recurrent_conn = SparseConn(conn=weights, enforce_dale=True)

        asc_amps = self._build_adaptation_amps(n_neuron, n_adapt, asc_amp)
        k = self._build_k_values(
            n_neuron=n_neuron,
            n_adapt=n_adapt,
            tau_adapt=tau_adapt,
            tau_adapt_min=tau_adapt_min,
            tau_adapt_max=tau_adapt_max,
        )
        self.neuron = GLIF3(
            n_neuron=n_neuron,
            v_threshold=v_threshold,
            v_reset=v_reset,
            tau=tau,
            tau_ref=tau_ref,
            asc_amps=asc_amps,
            k=k,
        )
        self.synapse = ExponentialPSC(
            n_neuron=n_neuron,
            tau_syn=tau_syn,
            linear=self.recurrent_conn,
        )

        self.rnn = RecurrentNN(
            neuron=self.neuron,
            synapse=self.synapse,
            step_mode="m",
            update_state_names=("neuron.v", "neuron.Iasc", "synapse.psc"),
            grad_checkpoint=grad_checkpoint,
            unroll=unroll,
            chunk_size=chunk_size,
            cpu_offload=cpu_offload,
        )

        self.output_linear = nn.Linear(n_neuron, output_dim, bias=True)
        self.output_scale = LearnableScale(scale=output_scale, bias=None)

        self.readout_alpha = float(
            torch.exp(torch.tensor(-self.dt / readout_tau)).item()
        )
        self.register_buffer("readout_filter_state", torch.zeros(1), persistent=False)
        self._state_initialized = False

    def _build_adaptation_amps(
        self,
        n_neuron: int,
        n_adapt: int,
        asc_amp: float,
    ) -> torch.Tensor:
        asc_amps = torch.zeros(n_neuron, 1, dtype=torch.float32)
        if n_adapt == 0:
            asc_amps[:, 0] = asc_amp
            return asc_amps
        if n_adapt == -1:
            asc_amps[: n_neuron // 2, 0] = asc_amp
            return asc_amps
        if n_adapt > 0:
            asc_amps[: min(n_adapt, n_neuron), 0] = asc_amp
        return asc_amps

    def _build_k_values(
        self,
        n_neuron: int,
        n_adapt: int,
        tau_adapt: float,
        tau_adapt_min: float | None,
        tau_adapt_max: float | None,
    ) -> torch.Tensor:
        k = torch.full((n_neuron, 1), 1.0 / tau_adapt, dtype=torch.float32)
        if tau_adapt_min is None or tau_adapt_max is None:
            return k
        if tau_adapt_min <= 0 or tau_adapt_max <= 0:
            raise ValueError("tau_adapt_min/tau_adapt_max must be positive")
        n_heter = (
            n_neuron if n_adapt == 0 else (n_neuron // 2 if n_adapt == -1 else n_adapt)
        )
        n_heter = max(0, min(n_neuron, n_heter))
        if n_heter == 0:
            return k
        taus = torch.empty(n_heter).uniform_(tau_adapt_min, tau_adapt_max)
        k[:n_heter, 0] = 1.0 / taus
        return k

    def get_response_window(self, T: int) -> slice:
        start = int(T * (1.0 - self.response_window))
        return slice(start, T)

    def _lowpass_filter(self, spikes: torch.Tensor) -> torch.Tensor:
        filtered = torch.zeros_like(spikes)
        filtered[0] = spikes[0]
        alpha = self.readout_alpha
        for t in range(1, spikes.shape[0]):
            filtered[t] = alpha * filtered[t - 1] + (1.0 - alpha) * spikes[t]
        return filtered

    def apply_dale_projection(self) -> None:
        if hasattr(self.recurrent_conn, "constrain"):
            self.recurrent_conn.constrain()

    def dale_violations(self) -> dict[str, float]:
        return sparseconn_dale_violations(self.recurrent_conn, n_e_neurons=self.n_e)

    def reset_state(self, batch_size: int, device: torch.device | None = None) -> None:
        if not self._state_initialized:
            functional.init_net_state(self.rnn, batch_size=batch_size, device=device)
            self._state_initialized = True
        functional.reset_net(self.rnn, batch_size=batch_size, device=device)

    def init_voltage(self, low: float = -70.0, high: float = -40.0) -> None:
        if not self._state_initialized:
            functional.init_net_state(self.rnn, batch_size=1)
            self._state_initialized = True
        uniform_v_(self.neuron, low=low, high=high, set_reset_value=True)

    def forward(
        self,
        x: torch.Tensor,
        *,
        reset_state: bool = True,
        return_sequence: bool = False,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Forward sequence through RSNN.

        Args:
            x: (T, batch, input_dim)
            reset_state: reset recurrent states before forward
            return_sequence: if True return (T, batch, output_dim)
                else average over response window to (batch, output_dim)
        """
        if x.ndim != 3:
            raise ValueError("x must be (T, batch, input_dim)")
        T, batch_size, _ = x.shape

        if reset_state:
            self.reset_state(batch_size=batch_size, device=x.device)

        x_scaled = self.input_scale(x)
        input_current = self.input_linear(x_scaled)

        with environ.context(dt=self.dt):
            spikes, rnn_states = self.rnn(input_current)

        filtered_spikes = self._lowpass_filter(spikes)
        out_dtype = self.output_linear.weight.dtype
        filtered_spikes_out = filtered_spikes.to(dtype=out_dtype)
        sequence_output = self.output_linear(self.output_scale(filtered_spikes_out))

        if return_sequence:
            output = sequence_output
        else:
            response_spikes = filtered_spikes_out[self.get_response_window(T)]
            avg_spikes = response_spikes.mean(dim=0)
            output = self.output_linear(self.output_scale(avg_spikes))

        states = {
            "spikes": spikes,
            "voltage": rnn_states["neuron.v"],
            "psc": rnn_states["synapse.psc"],
            "Iasc": rnn_states["neuron.Iasc"],
            "filtered_spikes": filtered_spikes,
            "sequence_output": sequence_output,
        }
        return output, states


input_dim = len(SYMBOLS) * cfg_obj.input_neurons_per_symbol

model = SingleLayerGLIFRSNN(
    input_dim=input_dim,
    output_dim=2,
    n_neuron=cfg_obj.n_neuron,
    n_adapt=cfg_obj.n_adapt,
    n_e_ratio=cfg_obj.n_e_ratio,
    tau=cfg_obj.tau_mem,
    tau_syn=cfg_obj.tau_syn,
    tau_ref=cfg_obj.tau_ref,
    v_threshold=cfg_obj.v_threshold,
    v_reset=cfg_obj.v_reset,
    asc_amp=cfg_obj.asc_amp,
    tau_adapt=700.0,
    tau_adapt_min=cfg_obj.tau_adapt_min,
    tau_adapt_max=cfg_obj.tau_adapt_max,
    dt=cfg_obj.dt,
    grad_checkpoint=cfg_obj.grad_checkpoint,
    chunk_size=cfg_obj.chunk_size,
).to(device)
model.init_voltage()

print(model)
print("grad_checkpoint:", getattr(model.rnn, "grad_checkpoint", None))
print("chunk_size:", getattr(model.rnn, "chunk_size", None))
print("initial input_scale.scale:", float(model.input_scale.scale.item()))


## 6) Forward pass + symbol aggregation
Sequence logits are timestep-level `(T,B,2)` and then reduced to symbol-level `(S,B,2)`.

In [ ]:
with torch.no_grad():
    seq_logits, states = model(x_batch, return_sequence=True)

symbol_logits = aggregate_symbol_logits(seq_logits, cfg_obj.symbol_ms)
print("sequence logits:", tuple(seq_logits.shape))
print("symbol logits:", tuple(symbol_logits.shape))
print("states[spikes]:", tuple(states["spikes"].shape))
print("states[voltage]:", tuple(states["voltage"].shape))
print("states[psc]:", tuple(states["psc"].shape))
print("states[Iasc]:", tuple(states["Iasc"].shape))


## 7) Compact training loop (BPTT)
Includes CE loss, firing-rate regularization, gradient clipping, optimizer step, and Dale projection.

In [ ]:
def compute_rate_regularization(spikes: torch.Tensor, dt: float, target_rate_hz: float) -> torch.Tensor:
    T, B, _ = spikes.shape
    duration_sec = T * dt / 1000.0
    rates = spikes.sum(dim=(0, 1)) / max(B * duration_sec, 1e-12)
    return ((rates - target_rate_hz) ** 2).mean()


optimizer = torch.optim.Adam(model.parameters(), lr=cfg_obj.lr)
metrics: list[dict[str, float | int]] = []
scale_history: list[float] = []

for it in range(1, cfg_obj.iterations + 1):
    model.train()
    x_train, y_train = generate_batch(cfg_obj, rng, device)

    optimizer.zero_grad(set_to_none=True)
    seq_logits, states = model(x_train, return_sequence=True)
    symbol_logits = aggregate_symbol_logits(seq_logits, cfg_obj.symbol_ms)

    loss_ce = F.cross_entropy(symbol_logits.reshape(-1, 2), y_train.reshape(-1))
    loss_rate = compute_rate_regularization(states["spikes"], cfg_obj.dt, cfg_obj.target_rate_hz)
    loss = loss_ce + cfg_obj.rate_reg_coeff * loss_rate

    loss.backward()
    if cfg_obj.max_grad_norm > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg_obj.max_grad_norm)
    optimizer.step()
    model.apply_dale_projection()

    pred = symbol_logits.argmax(dim=-1)
    row: dict[str, float | int] = {
        "iter": it,
        "train_loss": float(loss.item()),
        "train_ce": float(loss_ce.item()),
        "train_rate": float(loss_rate.item()),
        "train_symbol_acc": float((pred == y_train).float().mean().item()),
        "train_episode_success": float((pred == y_train).all(dim=0).float().mean().item()),
        "dale_violation_fraction": float(model.dale_violations()["fraction"]),
    }

    if it % cfg_obj.eval_every == 0 or it == cfg_obj.iterations:
        model.eval()
        eval_losses: list[float] = []
        eval_sym_accs: list[float] = []
        eval_ep_accs: list[float] = []

        with torch.no_grad():
            for _ in range(cfg_obj.eval_batches):
                x_eval, y_eval = generate_batch(cfg_obj, rng, device)
                eval_logits_seq, eval_states = model(x_eval, return_sequence=True)
                eval_symbol_logits = aggregate_symbol_logits(eval_logits_seq, cfg_obj.symbol_ms)

                eval_ce = F.cross_entropy(eval_symbol_logits.reshape(-1, 2), y_eval.reshape(-1))
                eval_rate = compute_rate_regularization(
                    eval_states["spikes"], cfg_obj.dt, cfg_obj.target_rate_hz
                )
                eval_loss = eval_ce + cfg_obj.rate_reg_coeff * eval_rate

                eval_pred = eval_symbol_logits.argmax(dim=-1)
                eval_losses.append(float(eval_loss.item()))
                eval_sym_accs.append(float((eval_pred == y_eval).float().mean().item()))
                eval_ep_accs.append(float((eval_pred == y_eval).all(dim=0).float().mean().item()))

        row.update(
            {
                "eval_loss": float(np.mean(eval_losses)),
                "eval_symbol_acc": float(np.mean(eval_sym_accs)),
                "eval_episode_success": float(np.mean(eval_ep_accs)),
            }
        )

    metrics.append(row)
    scale_history.append(float(model.input_scale.scale.item()))

metrics_df = pd.DataFrame(metrics)
metrics_df.tail()


## 8) Learnable input scale behavior
Input scale adapts to rescale effective drive into recurrent neurons under the current task/loss landscape.

In [ ]:
plot_scale_history(scale_history)
print("initial scale:", scale_history[0])
print("final scale:", scale_history[-1])


## 9) Evaluate and plot learning curves

In [ ]:
plot_training_curves(metrics_df)

final_row = metrics_df.iloc[-1].to_dict()
print({k: final_row[k] for k in ["iter", "train_loss", "train_symbol_acc", "train_episode_success"]})
if "eval_symbol_acc" in final_row:
    print({
        "eval_loss": final_row.get("eval_loss"),
        "eval_symbol_acc": final_row.get("eval_symbol_acc"),
        "eval_episode_success": final_row.get("eval_episode_success"),
    })


## 10) Checkpoint demo with memory snapshots
Use project checkpoint helpers so storage/restore behavior matches training scripts.

In [ ]:
run_dir = Path("../outputs/notebook_12ax_demo")
run_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = run_dir / "checkpoint_demo.pth"

best_sym_acc = float(metrics_df["train_symbol_acc"].max())
save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=int(metrics_df["iter"].max()),
    best_acc=best_sym_acc,
    path=ckpt_path,
)

ckpt_obj = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("checkpoint keys:", sorted(ckpt_obj.keys()))
print("has memories_rv:", "memories_rv" in ckpt_obj)
print("has memory_values:", "memory_values" in ckpt_obj)

loaded_epoch, loaded_best = load_checkpoint(
    path=ckpt_path,
    model=model,
    optimizer=optimizer,
    device=str(device),
    restore_memory_values=False,
)
print({"loaded_epoch": loaded_epoch, "loaded_best_acc": loaded_best})

_ = load_checkpoint(
    path=ckpt_path,
    model=model,
    optimizer=optimizer,
    device=str(device),
    restore_memory_values=True,
)
print("restore_memory_values=True applied for exact runtime-state restoration demo.")
print("Note: exact-state replay is sensitive to operation order and stochasticity.")


## 11) Optional state visualization
Quick spike raster subset for interpretability.

In [ ]:
with torch.no_grad():
    x_vis, _ = generate_batch(cfg_obj, rng, device)
    _, vis_states = model(x_vis, return_sequence=True)

plot_spike_raster(vis_states["spikes"], max_time=300, max_neurons=50, batch_index=0)


## 12) Recap and extension ideas
You now have an end-to-end 12AX notebook with explicit btorch operations and lean utility imports.

Try next:
- set `grad_checkpoint=False` and compare memory/runtime
- reduce or increase `chunk_size` in model construction
- change `input_neurons_per_symbol` and watch scaling behavior
- switch `profile` from `quick` to `paper` and compare outcomes
